# causal_direction_inference: analysis (v2, new pipeline)

Migrates this figure's analysis to `twinfer.inference.infer_with_twinfer_updated`,
including a new pipeline feature: `rescue_opposite_sign_pairs` (see
`infer_with_twinfer_updated.py`) -- for negative-feedback conditions like
`A_rep_B_B_to_A` (A represses B, B activates A) where the same-time
gene-gene correlation washes out to ~0 even though the two directional
cross-correlations are real and opposite-signed.

The actual compute is driven by standalone scripts (SLURM-friendly, with
per-task resumability), not `joblib.Parallel` in this notebook directly --
same pattern as `heterogeneity_vs_regulation/analysis_v2.ipynb`:

- `run_causal_direction_v2.py` -- Part A: 2-gene direction inference, 6 conditions.
- `run_cascade_v2.py` -- Part B: 5-gene linear cascade (real + remove_twin_structure control).
- `score_cascade_results.py` -- Part C: precision/recall/F1 for the cascade vs. ground truth.
- `run_grnboost2_gmm.py` -- Part D: GRNBoost2 baseline (GMM threshold only, run in the `grnboost310` env).
- `run_causal_direction_v2_slurm.sh` -- SLURM wrapper for Parts A+B.
- `aggregate_causal_direction_results.py` -- consolidates Part A's per-task JSONs into the CSVs `plot_v2.ipynb` reads.

This notebook demonstrates the pipeline on one replicate of each part
interactively, and shows how to run/aggregate the full batch.

In [ ]:
import sys
# [2026-09-30 commented out: modules are imported via dotted package paths (env.sh puts clean_code and clean_code/package on PYTHONPATH)]
# sys.path.insert(0, ".")
from paper_analysis.causal_direction_inference.run_causal_direction_v2 import (
    CONDITIONS,
    collect_tasks as collect_direction_tasks,
    process_replicate as process_direction_replicate,
)
from paper_analysis.causal_direction_inference.run_cascade_v2 import (
    collect_tasks as collect_cascade_tasks,
    run_cascade_replicate,
)

SIM_FOLDER = "/projects/b1255/yscher/Transcriptomic Distance/simulation_data/figure_3_simulations"

## Part A demo: one replicate of the motivating negative-feedback condition

In [ ]:
direction_tasks = collect_direction_tasks(SIM_FOLDER)
print(f"{len(direction_tasks)} total Part A tasks across {len(CONDITIONS)} conditions")

demo_path, demo_condition, demo_rep = next(
    t for t in direction_tasks if t[1] == "A_rep_B_B_to_A"
)
matrix_record, box_plot_record = process_direction_replicate(
    demo_path, demo_condition, demo_rep, n_cores=1, n_shuffles=2000
)
print("directed_edges:", matrix_record["directed_edges"])
print("rescued_from_no_regulation:", matrix_record["rescued_from_no_regulation"])
print("box_plot_record:", box_plot_record)

## Part B demo: the 5-gene cascade (real network + remove_twin_structure control)

In [ ]:
cascade_tasks = collect_cascade_tasks(SIM_FOLDER, "five_gene_cascade")
print(f"{len(cascade_tasks)} cascade replicate(s) available")

cascade_path, cascade_rep = cascade_tasks[0]
cascade_record = run_cascade_replicate(cascade_path, cascade_rep, n_cores=1, n_shuffles=2000)
print("real directed_edges:", cascade_record["real"]["directed_edges"])
print("random (remove_twin_structure) directed_edges:", cascade_record["random"]["directed_edges"])

## Running the full batch

Parts A+B (SLURM, `-A b1042 -p genomics`):
```bash
sbatch run_causal_direction_v2_slurm.sh
```
This runs all 120 Part A replicates + the cascade replicate(s), then calls
`aggregate_causal_direction_results.py` and `score_cascade_results.py`
automatically. Output lands under
`analysis_data/paper_analysis/causal_direction_inference/slurm_run/`.

Part D (GRNBoost2, separate `grnboost310` env, not SLURM-batched here since
there's currently only one cascade replicate):
```bash
/home/gzu5140/.conda/envs/grnboost310/bin/python3 run_grnboost2_gmm.py \
    --output-dir analysis_data/.../grnboost2_results
```

Both scripts are resumable -- safe to re-run; already-computed tasks are skipped.